###### PYSPARK

In [5]:
import sys
import pyspark
import pandas as pd

print("Python: ", sys.version)
print("Pyspark: ", pyspark.__version__)
print("Pandas: ", pd.__version__)

Python:  3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Pyspark:  4.2.0
Pandas:  3.0.6


In [1]:
from pyspark.sql import SparkSession

In [3]:
# spark = SparkSession.builder.appName("day1App").getOrCreate()

spark = (
    SparkSession.builder
    .appName("day1App")
    .master("local[*]")
    .getOrCreate()
)

In [4]:
print("spark version: ", spark.version)
print("application: ", spark.sparkContext.appName)

spark version:  4.2.0
application:  day1App


In [5]:
data = [
    (101, "Amisha", "IT", 60000),
    (102, "Urvi", "HR", 45000),
    (103, "Rahul", "IT", 75000),
    (104, "Neha", "Finance", 55000),
    (105, "Naga", "HR", 40000)
]

columns = ["ID", "Name", "Department", "Salary"]

df = spark.createDataFrame(data, columns)


In [6]:
df.show()

+---+------+----------+------+
| ID|  Name|Department|Salary|
+---+------+----------+------+
|101|Amisha|        IT| 60000|
|102|  Urvi|        HR| 45000|
|103| Rahul|        IT| 75000|
|104|  Neha|   Finance| 55000|
|105|  Naga|        HR| 40000|
+---+------+----------+------+



In [7]:
df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: long (nullable = true)



In [ ]:
df.count()   # It Returns the Number of Rows

5

In [10]:
df.select("Name", "Salary").show()

+------+------+
|  Name|Salary|
+------+------+
|Amisha| 60000|
|  Urvi| 45000|
| Rahul| 75000|
|  Neha| 55000|
|  Naga| 40000|
+------+------+



In [11]:
high_salary_df = df.filter(df.Salary > 50000)
high_salary_df.show()

+---+------+----------+------+
| ID|  Name|Department|Salary|
+---+------+----------+------+
|101|Amisha|        IT| 60000|
|103| Rahul|        IT| 75000|
|104|  Neha|   Finance| 55000|
+---+------+----------+------+



In [12]:
from pyspark.sql.functions import col

df.filter(col("Salary") > 50000).show()

+---+------+----------+------+
| ID|  Name|Department|Salary|
+---+------+----------+------+
|101|Amisha|        IT| 60000|
|103| Rahul|        IT| 75000|
|104|  Neha|   Finance| 55000|
+---+------+----------+------+



In [13]:
df.show()

+---+------+----------+------+
| ID|  Name|Department|Salary|
+---+------+----------+------+
|101|Amisha|        IT| 60000|
|102|  Urvi|        HR| 45000|
|103| Rahul|        IT| 75000|
|104|  Neha|   Finance| 55000|
|105|  Naga|        HR| 40000|
+---+------+----------+------+



In [14]:
df_with_ctc = df.withColumn(
    "annual_ctc",
    col("Salary") * 12
)

df_with_ctc.show()

+---+------+----------+------+----------+
| ID|  Name|Department|Salary|annual_ctc|
+---+------+----------+------+----------+
|101|Amisha|        IT| 60000|    720000|
|102|  Urvi|        HR| 45000|    540000|
|103| Rahul|        IT| 75000|    900000|
|104|  Neha|   Finance| 55000|    660000|
|105|  Naga|        HR| 40000|    480000|
+---+------+----------+------+----------+



In [15]:
df.show()

+---+------+----------+------+
| ID|  Name|Department|Salary|
+---+------+----------+------+
|101|Amisha|        IT| 60000|
|102|  Urvi|        HR| 45000|
|103| Rahul|        IT| 75000|
|104|  Neha|   Finance| 55000|
|105|  Naga|        HR| 40000|
+---+------+----------+------+



In [17]:
df_with_bonus = df_with_ctc.withColumn(
    "Bonus",
    col("annual_ctc") * 0.8
)

df_with_bonus.show()

+---+------+----------+------+----------+--------+
| ID|  Name|Department|Salary|annual_ctc|   Bonus|
+---+------+----------+------+----------+--------+
|101|Amisha|        IT| 60000|    720000|576000.0|
|102|  Urvi|        HR| 45000|    540000|432000.0|
|103| Rahul|        IT| 75000|    900000|720000.0|
|104|  Neha|   Finance| 55000|    660000|528000.0|
|105|  Naga|        HR| 40000|    480000|384000.0|
+---+------+----------+------+----------+--------+



In [18]:
from pyspark.sql.functions import count, avg, sum

In [19]:
department_summary = df_with_bonus.groupBy("Department").agg(
    count("*").alias("employee_count"),
    sum("annual_ctc").alias("total_yearly_salay"),
    avg("Salary").alias("Average_Monthly_Salary")
)

department_summary.show()

+----------+--------------+------------------+----------------------+
|Department|employee_count|total_yearly_salay|Average_Monthly_Salary|
+----------+--------------+------------------+----------------------+
|        IT|             2|           1620000|               67500.0|
|        HR|             2|           1020000|               42500.0|
|   Finance|             1|            660000|               55000.0|
+----------+--------------+------------------+----------------------+

